# Data Pipeline & Preprocessing

## Purpose
Clean, validate, and prepare the raw HR dataset
for machine learning modeling.

## Input
data/raw/employee_performance_raw.csv

## Output
- data/processed/employee_processed.csv
- models/preprocessing_pipeline.joblib

## Steps
1. Load raw data
2. Validate data quality
3. Separate columns by type
4. Build preprocessing pipeline
5. Apply pipeline
6. Save outputs

In [1]:
# ── Import Libraries ──

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# sklearn preprocessing tools
from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute        import SimpleImputer, KNNImputer

# For saving the pipeline
import joblib

print("All libraries imported")

All libraries imported


In [2]:
# ── Load Raw Data ──

# Build file paths
notebook_dir  = os.getcwd()
project_root  = os.path.dirname(notebook_dir)

raw_data_path       = os.path.join(project_root, 'data', 'raw',
                                    'employee_performance_raw.csv')
processed_data_path = os.path.join(project_root, 'data', 'processed',
                                    'employee_processed.csv')
pipeline_save_path  = os.path.join(project_root, 'models',
                                    'preprocessing_pipeline.joblib')

# Create output folders if they don't exist
os.makedirs(os.path.join(project_root, 'data', 'processed'), exist_ok=True)
os.makedirs(os.path.join(project_root, 'models'), exist_ok=True)

# Load the raw data
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded successfully")
print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]}")
print(f"\nColumns available:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:>2}. {col}") 

Raw data loaded successfully
Rows    : 2,600
Columns : 37

Columns available:
 1. EmployeeID
 2. Department
 3. JobRole
 4. JobLevel
 5. ManagerID
 6. LocationType
 7. Age
 8. Gender
 9. EducationLevel
10. YearsAtCompany
11. YearsSinceLastPromotion
12. PerformanceRating
13. HistoricalRatingAvg
14. CalibrationAdjustedFlag
15. SelfRating
16. PeerAvgRating
17. SubordinateAvgRating
18. Overall360Score
19. SelfOtherGap
20. Leadership360
21. Collaboration360
22. OKRCompletionPct
23. NumOKRsAssigned
24. WeightedGoalAttainment
25. EngagementScore
26. JobSatisfaction
27. WorkLifeBalanceRating
28. BurnoutRisk
29. TrainingHoursLastYear
30. OvertimeHoursMonthly
31. AbsenteeismDays
32. AvgMonthlyHours
33. ProjectsHandled
34. PercentSalaryHikeLast
35. BonusPayoutPct
36. HighPotentialFlag
37. PIPHistoryFlag


In [3]:
# ── Data Validation ── 

print("-" * 55)
print("DATA VALIDATION REPORT")
print("-" * 55)

df = df_raw.copy()  # Work on a copy, keep original safe
issues_found = 0

# ── Duplicate Employee IDs ──
duplicates = df['EmployeeID'].duplicated().sum()
if duplicates > 0:
    print(f"ISSUE: {duplicates} duplicate Employee IDs found")
    df = df.drop_duplicates(subset='EmployeeID')
    issues_found += 1
else:
    print(f"No duplicate Employee IDs")

# ── Valid Range Checks ──
# Define valid ranges for numeric fields
range_rules = {
    'PerformanceRating'     : (1, 5),
    'HistoricalRatingAvg'   : (1, 5),
    'SelfRating'            : (1, 5),
    'Overall360Score'       : (1, 5),
    'OKRCompletionPct'      : (0, 100),
    'WeightedGoalAttainment': (0, 100),
    'EngagementScore'       : (0, 100),
    'Age'                   : (18, 70),
    'AbsenteeismDays'       : (0, 365),
    'TrainingHoursLastYear' : (0, 500),
}

for col, (lo, hi) in range_rules.items():
    if col in df.columns:
        out_of_range = ((df[col] < lo) | (df[col] > hi)).sum()
        if out_of_range > 0:
            print(f"ISSUE: {out_of_range} out-of-range values in {col} "
                  f"(valid: {lo}–{hi}) → clipping to valid range")
            df[col] = df[col].clip(lo, hi)
            issues_found += 1
        else:
            print(f"{col:<35} range OK ({lo}–{hi})")

# ── Outlier Capping for Hours Fields ──
# Use IQR method: anything beyond Q3 + 3×IQR is extreme outlier
hours_cols = ['OvertimeHoursMonthly', 'AvgMonthlyHours',
              'TrainingHoursLastYear']

print(f"\nOutlier check (IQR method):")
for col in hours_cols:
    if col in df.columns:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_limit = Q3 + 3 * IQR
        outliers = (df[col] > upper_limit).sum()
        if outliers > 0:
            df[col] = df[col].clip(upper=upper_limit)
            print(f"{col}: {outliers} outliers capped at {upper_limit:.1f}")
            issues_found += 1
        else:
            print(f"{col}: no extreme outliers")

# ── Missing Value Summary ──
print(f"\nMissing values summary:")
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    for col, count in missing_cols.items():
        pct = count / len(df) * 100
        print(f"{col:<35} {count:>4} missing ({pct:.1f}%)")
else:
    print(f"No missing values found")

print(f"\n{'-'*55}")
print(f"Total issues found and fixed        : {issues_found}")
print(f"Rows after validation               : {df.shape[0]:,}")
print(f"{'-'*55}")
print(f"Validation complete") 

-------------------------------------------------------
DATA VALIDATION REPORT
-------------------------------------------------------
No duplicate Employee IDs
PerformanceRating                   range OK (1–5)
HistoricalRatingAvg                 range OK (1–5)
SelfRating                          range OK (1–5)
Overall360Score                     range OK (1–5)
OKRCompletionPct                    range OK (0–100)
WeightedGoalAttainment              range OK (0–100)
EngagementScore                     range OK (0–100)
Age                                 range OK (18–70)
AbsenteeismDays                     range OK (0–365)
TrainingHoursLastYear               range OK (0–500)

Outlier check (IQR method):
OvertimeHoursMonthly: 18 outliers capped at 57.3
AvgMonthlyHours: no extreme outliers
TrainingHoursLastYear: no extreme outliers

Missing values summary:
PeerAvgRating                        150 missing (5.8%)
SubordinateAvgRating                1938 missing (74.5%)
Overall360Score      

In [4]:
# ── Define Column Groups ──

# ── Columns to DROP completely ──
# Reason: ID fields or too many unique values for ML
DROP_COLS = [
    'EmployeeID',    # Just an identifier - no signal
    'ManagerID',     # 200 unique values - too many categories
    'JobRole',       # Redundant with Department + JobLevel
]

# ── NUMERIC columns ──
# Will be: imputed with KNN → scaled with StandardScaler
NUMERIC_COLS = [
    'Age',
    'YearsAtCompany',
    'YearsSinceLastPromotion',
    'HistoricalRatingAvg',
    'SelfRating',
    'PeerAvgRating',
    'SubordinateAvgRating',
    'Overall360Score',
    'SelfOtherGap',
    'Leadership360',
    'Collaboration360',
    'OKRCompletionPct',
    'NumOKRsAssigned',
    'WeightedGoalAttainment',
    'EngagementScore',
    'JobSatisfaction',
    'WorkLifeBalanceRating',
    'TrainingHoursLastYear',
    'OvertimeHoursMonthly',
    'AbsenteeismDays',
    'AvgMonthlyHours',
    'ProjectsHandled',
    'PercentSalaryHikeLast',
    'BonusPayoutPct',
]

# ── ORDINAL columns ──
# Has a natural order: IC1 < IC2 < IC3 < M1 < M2 < Director
# Will be: imputed → encoded as 0,1,2,3,4,5
ORDINAL_COLS       = ['JobLevel']
ORDINAL_CATEGORIES = [['IC1', 'IC2', 'IC3', 'M1', 'M2', 'Director']]

# ── NOMINAL columns ──
# No natural order between categories
# Will be: imputed → one-hot encoded (0s and 1s)
NOMINAL_COLS = [
    'Department',
    'LocationType',
    'Gender',
    'EducationLevel',
    'BurnoutRisk',
]

# ── BINARY columns ──
# Already 0 or 1 — just need missing values filled
BINARY_COLS = [
    'CalibrationAdjustedFlag',
    'HighPotentialFlag',
    'PIPHistoryFlag',
]

# ── TARGET column (what we want to predict) ──
# This is KEPT SEPARATE — not processed by the pipeline
TARGET_COL = 'PerformanceRating'

# ── Verify all columns are accounted for ──
all_defined = (
    DROP_COLS + NUMERIC_COLS + ORDINAL_COLS +
    NOMINAL_COLS + BINARY_COLS + [TARGET_COL]
)

all_in_df = df.columns.tolist()

missing_from_definition = [c for c in all_in_df
                            if c not in all_defined]
extra_in_definition     = [c for c in all_defined
                            if c not in all_in_df]

print("-" * 55)
print("COLUMN GROUPING SUMMARY")
print("-" * 55)
print(f"DROP columns    : {len(DROP_COLS)}")
print(f"NUMERIC columns : {len(NUMERIC_COLS)}")
print(f"ORDINAL columns : {len(ORDINAL_COLS)}")
print(f"NOMINAL columns : {len(NOMINAL_COLS)}")
print(f"BINARY columns  : {len(BINARY_COLS)}")
print(f"TARGET column   : {TARGET_COL}")
print(f"─────────────────────────────────────")
print(f"Total accounted : {len(all_defined)}")
print(f"Total in dataset: {len(all_in_df)}")

if missing_from_definition:
    print(f"\nColumns in data but not defined:")
    for c in missing_from_definition:
        print(f" → {c}")
else:
    print(f"\nAll columns accounted for")

if extra_in_definition:
    print(f"\nColumns defined but not in data:")
    for c in extra_in_definition:
        print(f" → {c}") 

-------------------------------------------------------
COLUMN GROUPING SUMMARY
-------------------------------------------------------
DROP columns    : 3
NUMERIC columns : 24
ORDINAL columns : 1
NOMINAL columns : 5
BINARY columns  : 3
TARGET column   : PerformanceRating
─────────────────────────────────────
Total accounted : 37
Total in dataset: 37

All columns accounted for


In [5]:
# ── Build Preprocessing Pipeline ──

# ── Build individual transformers ──

# For NUMERIC columns:
# KNN imputer fills missing values using 5 nearest neighbors
# StandardScaler brings all numbers to same scale
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  StandardScaler())
])

# For ORDINAL columns:
# Fill missing with most common value
# Then encode as ordered numbers (IC1=0, IC2=1 etc.)
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ORDINAL_CATEGORIES,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# For NOMINAL columns:
# Fill missing with most common value
# Then create one 0/1 column per category
# drop='first' removes one column to avoid redundancy
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False,
        drop='first'
    ))
])

# For BINARY columns:
# Just fill missing values with most common (0 or 1)
# No encoding needed — already numeric
binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# ── Combine into one ColumnTransformer ──
# This applies the right transformer to the right columns
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, NUMERIC_COLS),
        ('ordinal', ordinal_transformer, ORDINAL_COLS),
        ('nominal', nominal_transformer, NOMINAL_COLS),
        ('binary',  binary_transformer,  BINARY_COLS),
    ],
    remainder='drop'  # Drop any columns not listed above
)

print("Preprocessing pipeline built")
print()
print("Pipeline structure:")
print("numeric  → KNNImputer → StandardScaler")
print("ordinal  → SimpleImputer → OrdinalEncoder")
print("nominal  → SimpleImputer → OneHotEncoder")
print("binary   → SimpleImputer")
print() 
print(f"Input columns  : {len(NUMERIC_COLS + ORDINAL_COLS + NOMINAL_COLS + BINARY_COLS)}") 

Preprocessing pipeline built

Pipeline structure:
numeric  → KNNImputer → StandardScaler
ordinal  → SimpleImputer → OrdinalEncoder
nominal  → SimpleImputer → OneHotEncoder
binary   → SimpleImputer

Input columns  : 33


In [6]:
# ── Prepare Features and Target ──

# ── Remove leakage columns from features ──
# Leakage = columns that would not exist BEFORE the rating
# is assigned in real life
# PercentSalaryHikeLast and BonusPayoutPct are calculated
# AFTER performance ratings — so they cannot be used to predict ratings
LEAKAGE_COLS = [
    'PercentSalaryHikeLast',  # Decided after rating
    'BonusPayoutPct',         # Decided after rating
]

# Remove leakage columns from NUMERIC_COLS for modeling
NUMERIC_COLS_MODEL = [c for c in NUMERIC_COLS
                      if c not in LEAKAGE_COLS]

print("Leakage columns removed from features:")
for col in LEAKAGE_COLS:
    print(f" → {col} (calculated after performance rating)")

# ── Create feature matrix X ──
# All columns except target and drop columns
feature_cols = NUMERIC_COLS_MODEL + ORDINAL_COLS + NOMINAL_COLS + BINARY_COLS
X = df[feature_cols].copy()

# ── Create target variable y ──
y_regression     = df[TARGET_COL].copy()     # 1.0 to 5.0 (for regression)
y_classification = (df[TARGET_COL] >= 4.0).astype(int)  # 0 or 1 (for classification)

# Add target columns to df for reference
df['HighPerformer'] = y_classification

print(f"\nFeatures and target prepared")
print(f"Feature matrix shape  : {X.shape}")
print(f"Feature columns       : {len(feature_cols)}")
print(f"\nTarget: PerformanceRating (regression)")
print(f"Min: {y_regression.min()}  Max: {y_regression.max()}  "
      f"Mean: {y_regression.mean():.2f}")
print(f"\nTarget: HighPerformer (classification)")
print(f"High Performers (1) : {y_classification.sum():,} "
      f"({y_classification.mean()*100:.1f}%)")
print(f"Standard     (0)    : {(y_classification==0).sum():,} "
      f"({(y_classification==0).mean()*100:.1f}%)")
print(f"\nClass imbalance ratio : "
      f"{(y_classification==0).sum() / y_classification.sum():.1f}:1")
print(f"(Standard : High Performer)") 

Leakage columns removed from features:
 → PercentSalaryHikeLast (calculated after performance rating)
 → BonusPayoutPct (calculated after performance rating)

Features and target prepared
Feature matrix shape  : (2600, 31)
Feature columns       : 31

Target: PerformanceRating (regression)
Min: 1.0  Max: 5.0  Mean: 2.99

Target: HighPerformer (classification)
High Performers (1) : 541 (20.8%)
Standard     (0)    : 2,059 (79.2%)

Class imbalance ratio : 3.8:1
(Standard : High Performer)


In [7]:
# ── Apply the Pipeline ──

# Rebuild preprocessor with non-leakage columns
preprocessor_model = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, NUMERIC_COLS_MODEL),
        ('ordinal', ordinal_transformer, ORDINAL_COLS),
        ('nominal', nominal_transformer, NOMINAL_COLS),
        ('binary',  binary_transformer,  BINARY_COLS),
    ],
    remainder='drop'
)

# Apply pipeline to feature matrix
# fit_transform = learn from data AND apply transformation
print("Applying preprocessing pipeline...")
X_processed = preprocessor_model.fit_transform(X)

# ── Get output column names ──
# After encoding, column names change
# (e.g. Department becomes Department_Sales, Department_HR etc.)

# Numeric column names stay the same
numeric_feature_names = NUMERIC_COLS_MODEL

# Ordinal column names stay the same
ordinal_feature_names = ORDINAL_COLS

# Nominal columns expand into multiple columns after OneHotEncoding
nominal_feature_names = (
    preprocessor_model
    .named_transformers_['nominal']
    .named_steps['encoder']
    .get_feature_names_out(NOMINAL_COLS)
    .tolist()
)

# Binary column names stay the same
binary_feature_names = BINARY_COLS

# Combine all feature names in correct order
all_feature_names = (
    numeric_feature_names +
    ordinal_feature_names +
    nominal_feature_names +
    binary_feature_names
)

# Convert to DataFrame for readability
X_processed_df = pd.DataFrame(
    X_processed,
    columns=all_feature_names
)

print(f"Pipeline applied successfully")
print(f"\nInput shape  : {X.shape}")
print(f"Output shape : {X_processed_df.shape}")
print(f"New columns from OneHotEncoding:")
for col in nominal_feature_names:
    print(f" → {col}")
print(f"\nSample of processed data (first 3 rows, first 8 cols):")
print(X_processed_df.iloc[:3, :8].round(3).to_string()) 

Applying preprocessing pipeline...
Pipeline applied successfully

Input shape  : (2600, 31)
Output shape : (2600, 43)
New columns from OneHotEncoding:
 → Department_Engineering
 → Department_Finance
 → Department_HR
 → Department_Marketing
 → Department_Operations
 → Department_Product
 → Department_Sales
 → LocationType_On-Site
 → LocationType_Remote
 → Gender_Male
 → Gender_Non-Binary
 → EducationLevel_Bachelor's
 → EducationLevel_High School
 → EducationLevel_Master's
 → EducationLevel_PhD
 → BurnoutRisk_Low
 → BurnoutRisk_Medium

Sample of processed data (first 3 rows, first 8 cols):
     Age  YearsAtCompany  YearsSinceLastPromotion  HistoricalRatingAvg  SelfRating  PeerAvgRating  SubordinateAvgRating  Overall360Score
0 -0.484          -0.654                   -0.503               -1.007      -1.891         -0.846                -1.196           -1.330
1 -1.652          -0.000                   -0.293               -0.747       0.469         -0.578                 1.017            

In [8]:
# ── Create Final Processed Dataset ──

# We save TWO versions:

# VERSION 1: Human readable (for Power BI and EDA)
# Keep original column names + add target columns
df_for_powerbi = df.copy()

# Fill remaining missing values for the Power BI version
# ffill = forward fill (use previous row's value)
# bfill = backward fill (use next row's value, catches any remaining)
df_for_powerbi = df_for_powerbi.ffill().bfill()

# VERSION 2: ML ready (for modeling in Phase 7)
# Processed features + target
df_ml_ready = X_processed_df.copy()
df_ml_ready['PerformanceRating'] = y_regression.values
df_ml_ready['HighPerformer']     = y_classification.values
df_ml_ready['EmployeeID']        = df['EmployeeID'].values

print(f"Two versions of processed data created")
print(f"\nVersion 1 — Power BI / EDA version:")
print(f"Shape    : {df_for_powerbi.shape}")
print(f"Contains : Original column names, readable values")
print(f"\nVersion 2 — ML ready version:")
print(f"Shape    : {df_ml_ready.shape}")
print(f"Contains : Processed numbers, encoded categories")
print(f"\nMissing values in Power BI version:")
print(f"{df_for_powerbi.isnull().sum().sum()} remaining missing values")
print(f"\nMissing values in ML version:")
print(f"{df_ml_ready.isnull().sum().sum()} remaining missing values") 

Two versions of processed data created

Version 1 — Power BI / EDA version:
Shape    : (2600, 38)
Contains : Original column names, readable values

Version 2 — ML ready version:
Shape    : (2600, 46)
Contains : Processed numbers, encoded categories

Missing values in Power BI version:
0 remaining missing values

Missing values in ML version:
0 remaining missing values


In [9]:
# ── Save All Outputs ──

# ── Save Power BI / EDA version ──
powerbi_path = os.path.join(
    project_root, 'data', 'processed', 'employee_processed.csv'
)
df_for_powerbi.to_csv(powerbi_path, index=False)
print(f"Power BI dataset saved:")
print(f"{powerbi_path}")

# ── Save ML ready version ──
ml_path = os.path.join(
    project_root, 'data', 'processed', 'employee_ml_ready.csv'
)
df_ml_ready.to_csv(ml_path, index=False)
print(f"\nML ready dataset saved:")
print(f"{ml_path}")

# ── Save The preprocessing pipeline ──
# This is the most important save for automation
# Phase 11 will load this and use it every month
joblib.dump(preprocessor_model, pipeline_save_path)
print(f"\nPreprocessing pipeline saved:")
print(f"{pipeline_save_path}")

# ── Save feature column names for automation ──
# Phase 11 needs to know which columns to use
import json
feature_cols_path = os.path.join(project_root, 'models', 'feature_cols.json')
with open(feature_cols_path, 'w') as f:
    json.dump(feature_cols, f)
print(f"\nFeature column names saved:")
print(f"{feature_cols_path}")

print(f"\n{'-'*55}")
print(f"ALL OUTPUTS SAVED SUCCESSFULLY")
print(f"{'-'*55}")
print(f"\ndata/processed/employee_processed.csv  → Power BI")
print(f"data/processed/employee_ml_ready.csv   → ML Model")
print(f"models/preprocessing_pipeline.joblib   → Automation")
print(f"models/feature_cols.json               → Automation") 

Power BI dataset saved:
C:\Users\ganti_kvd0xe3\OneDrive\Sathya\employee_performance_analytics\data\processed\employee_processed.csv

ML ready dataset saved:
C:\Users\ganti_kvd0xe3\OneDrive\Sathya\employee_performance_analytics\data\processed\employee_ml_ready.csv

Preprocessing pipeline saved:
C:\Users\ganti_kvd0xe3\OneDrive\Sathya\employee_performance_analytics\models\preprocessing_pipeline.joblib

Feature column names saved:
C:\Users\ganti_kvd0xe3\OneDrive\Sathya\employee_performance_analytics\models\feature_cols.json

-------------------------------------------------------
ALL OUTPUTS SAVED SUCCESSFULLY
-------------------------------------------------------

data/processed/employee_processed.csv  → Power BI
data/processed/employee_ml_ready.csv   → ML Model
models/preprocessing_pipeline.joblib   → Automation
models/feature_cols.json               → Automation


In [10]:
# ── Final Verification ──

print("-" * 55)
print("FINAL VERIFICATION")
print("-" * 55)

# Check all files exist
files_to_check = {
    'Power BI CSV'   : powerbi_path,
    'ML ready CSV'   : ml_path,
    'Pipeline'       : pipeline_save_path,
    'Feature cols'   : feature_cols_path,
}

all_good = True
for name, path in files_to_check.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024  # KB
    status = "Accuracy" if exists else "Error"
    print(f"{status} {name:<20} {size:>8.1f} KB")
    if not exists:
        all_good = False

# Reload and verify CSVs
df_verify = pd.read_csv(powerbi_path)
print(f"\nPower BI CSV check:")
print(f"Rows    : {df_verify.shape[0]:,}")
print(f"Columns : {df_verify.shape[1]}")
print(f"Missing : {df_verify.isnull().sum().sum()}")

# Reload and verify pipeline
pipeline_verify = joblib.load(pipeline_save_path)
print(f"\nPipeline check:")
print(f"Type    : {type(pipeline_verify).__name__}")
# transformers returns (name, transformer, columns) — 3 values
# so we unpack all 3
print(f"Steps   : {[name for name, _, cols in pipeline_verify.transformers]}")

print(f"\n{'-'*55}")
if all_good:
    print(f"All files saved and verified")
else:
    print(f"Some files are missing — check errors above")
print("-" * 55) 

-------------------------------------------------------
FINAL VERIFICATION
-------------------------------------------------------
Accuracy Power BI CSV            485.9 KB
Accuracy ML ready CSV           1354.0 KB
Accuracy Pipeline                511.4 KB
Accuracy Feature cols              0.6 KB

Power BI CSV check:
Rows    : 2,600
Columns : 38
Missing : 0

Pipeline check:
Type    : ColumnTransformer
Steps   : ['numeric', 'ordinal', 'nominal', 'binary']

-------------------------------------------------------
All files saved and verified
-------------------------------------------------------
